# Weeks 9–12 Reflection Code

All of the code behind the Week 9–12 reflections, one section per week. The written answers are in `Kyle_Reynolds_Weeks_9-12_Reflections.pdf`. Week 10 had no reflection, so it has no section here.

Cells run top to bottom. Each week starts its own `rng` from seed 0, so the numbers match the reflection.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

## Week 9 — Standard errors under heteroskedasticity and correlated errors

The true model is always Y = 1 + 2X + error. Each question compares the real spread of the coefficient (from simulating many fresh datasets) with what OLS or the bootstrap reports from one dataset.

### Q1. Use a simulation to estimate the standard deviation of the coefficient when there is heteroskedasticity, and compare it to statsmodels OLS

The noise in Y grows with |X| (error sd = 2|X|). 1,000 datasets of 1,000 rows.

In [2]:
rng = np.random.default_rng(0)


def ols_se(x, y, cov_type="nonrobust"):
    return sm.OLS(y, sm.add_constant(x)).fit(cov_type=cov_type).bse[1]


def slope(x, y):
    return np.polyfit(x, y, 1)[0]


def sim_sd(make_data, n_datasets):
    """sd of the coef across many fresh datasets -- the real answer."""
    return np.std([slope(*make_data()) for _ in range(n_datasets)], ddof=1)


def hetero_data(n=1_000):
    x = rng.normal(0, 1, n)
    y = 1 + 2 * x + rng.normal(0, 1, n) * 2 * np.abs(x)   # error sd = 2|X|
    return x, y


x, y = hetero_data()
print("Q1 heteroskedastic errors:")
print(f"  simulated sd {sim_sd(hetero_data, 1_000):.4f}   OLS SE {ols_se(x, y):.4f}   "
      f"robust SE {ols_se(x, y, 'HC1'):.4f}")

Q1 heteroskedastic errors:
  simulated sd 0.1042   OLS SE 0.0647   robust SE 0.1127


### Q2. Use a simulation to estimate the standard deviation of the coefficient when errors are highly correlated, and compare it to statsmodels OLS

X and the error are both AR(1) series: each value is `rho` times the last plus new noise. 1,000 datasets of 100 rows per `rho`, with the OLS SE and a row bootstrap from one dataset.

In [3]:
def make_error(rho, n):
    """Each value is rho times the last plus new noise, scaled to variance 1."""
    err = np.empty(n)
    err[0] = rng.normal(0, 1)
    for i in range(1, n):
        err[i] = rho * err[i - 1] + np.sqrt(1 - rho**2) * rng.normal(0, 1)
    return err


def correlated_data(rho, n):
    x = make_error(rho, n)
    return x, 1 + 2 * x + make_error(rho, n)


def bootstrap_sd(x, y, block=1, n_boot=300):
    """Resample runs of `block` neighbouring rows (block = 1 is the usual bootstrap)."""
    n = len(x)
    coefs = []
    for _ in range(n_boot):
        starts = rng.integers(0, n - block + 1, n // block)
        rows = (starts[:, None] + np.arange(block)).ravel()
        coefs.append(slope(x[rows], y[rows]))
    return np.std(coefs, ddof=1)


print("Q2 correlated errors, 100 rows:")
print(f"{'rho':>5} {'simulated':>10} {'OLS':>7} {'bootstrap':>10}")
for rho in [0.0, 0.5, 0.9, 0.99]:
    x, y = correlated_data(rho, 100)
    sd = sim_sd(lambda: correlated_data(rho, 100), 1_000)
    print(f"{rho:5.2f} {sd:10.4f} {ols_se(x, y):7.4f} {bootstrap_sd(x, y):10.4f}")

Q2 correlated errors, 100 rows:
  rho  simulated     OLS  bootstrap
 0.00     0.1047  0.0967     0.0936
 0.50     0.1293  0.0949     0.0913
 0.90     0.2861  0.1312     0.1477
 0.99     0.5747  0.1149     0.1003


### Q3. Show that with highly correlated errors the bootstrap standard deviation might not match a full simulation

At `rho = 0.99`, compare the simulation with the row bootstrap and a block bootstrap (100 blocks of neighbouring rows) as the data grows. Takes about 20 seconds.

In [ ]:
print("   rho = 0.99, more data:")
print(f"{'rows':>8} {'simulated':>10} {'row boot':>9} {'block boot':>11}")
for n in [100, 10_000, 100_000]:
    x, y = correlated_data(0.99, n)
    sd = sim_sd(lambda: correlated_data(0.99, n), 200)
    block = max(n // 100, 1)   # 100 blocks
    print(f"{n:8,d} {sd:10.4f} {bootstrap_sd(x, y):9.4f} {bootstrap_sd(x, y, block):11.4f}")

   rho = 0.99, more data:
    rows  simulated  row boot  block boot
     100     0.5581    0.3551      0.3538
  10,000     0.1012    0.0108      0.0747


## Week 11 — Event studies

### Q1. Build an event study where the value, derivative, and second derivative all change at the event, and compare (a) a value-only model with (b) a value, derivative, and second derivative model

Time `s` is measured from the event at `s = 0`. After the event the level jumps by 2, the slope by 0.1, and the curvature by 0.002. Model (b) is tested with a joint F-test on its three event terms. Each model runs on 1,000 datasets with the event and 1,000 without.

In [ ]:
rng = np.random.default_rng(0)

LEVEL, SLOPE, CURVE = 2, 0.1, 0.002      # true jumps after the event


def trend_data(n=200, event=100, effect=True):
    s = np.arange(n) - event              # time measured from the event
    post = (s >= 0).astype(int)
    y = 10 + 0.05 * s + rng.normal(0, 1, n)
    if effect:
        y += post * (LEVEL + SLOPE * s + CURVE * s**2)
    return pd.DataFrame({"y": y, "s": s, "post": post})


def value_only(df):
    """(a) a single level jump on top of a straight-line trend."""
    fit = smf.ols("y ~ s + post", df).fit()
    return fit.params["post"], fit.pvalues["post"] < 0.05


def all_three(df):
    """(b) jumps in level, slope, and curvature, tested together with an F-test."""
    fit = smf.ols("y ~ s + post + post:s + post:I(s**2)", df).fit()
    joint = fit.f_test("post = 0, post:s = 0, post:I(s ** 2) = 0")
    return fit.params["post"], joint.pvalue < 0.05


def compare(n_sims=1_000):
    rows = []
    for name, model in [("(a) value only", value_only), ("(b) all three", all_three)]:
        real = np.array([model(trend_data()) for _ in range(n_sims)])
        null = np.array([model(trend_data(effect=False)) for _ in range(n_sims)])
        rows.append({"model": name,
                     "mean level est": real[:, 0].mean(),
                     "sd level est": real[:, 0].std(ddof=1),
                     "detected (real)": real[:, 1].mean(),
                     "false pos (null)": null[:, 1].mean()})
    return pd.DataFrame(rows).set_index("model").round(3)


print(f"Q1 true jumps: level {LEVEL}, slope {SLOPE}, curvature {CURVE}")
df = trend_data()
fit_a = smf.ols("y ~ s + post", df).fit()
fit_b = smf.ols("y ~ s + post + post:s + post:I(s**2)", df).fit()
print("  (a) one dataset:", fit_a.params.round(3).to_dict())
print("  (b) one dataset:", fit_b.params.round(4).to_dict())
print("\n  1,000 datasets each:")
print(compare().to_string())

### Q2. Three groups with different starting values jump by the same amount at a shared event: combine group fixed effects with an event study

Baselines 0, 5, and 10, a shared trend of 0.1 per period, and a jump of 3 at period 50. `C(group)` gives each group its own intercept (the fixed effects), `t` is the shared trend, and the coefficient on `post` is the common event effect.

In [ ]:
def group_data(n=100, event=50, jump=3):
    t = np.arange(n)
    frames = [pd.DataFrame({"group": g, "t": t, "post": (t >= event).astype(int),
                            "y": base + 0.1 * t + jump * (t >= event) + rng.normal(0, 1, n)})
              for g, base in zip("ABC", [0, 5, 10])]
    return pd.concat(frames, ignore_index=True)


df = group_data()
pooled = smf.ols("y ~ t + post", df).fit()
fe = smf.ols("y ~ C(group) + t + post", df).fit()
print("Q2 true jump 3, baselines 0 / 5 / 10")
print(f"  no group FE:   post = {pooled.params['post']:.3f} (SE {pooled.bse['post']:.3f})")
print(f"  with group FE: post = {fe.params['post']:.3f} (SE {fe.bse['post']:.3f})")
print("  group effects:", fe.params.filter(like="group").round(2).to_dict(),
      f"(group A intercept {fe.params['Intercept']:.2f})")

## Week 12 — Differences-in-differences when prior trends don't hold

### Construct a dataset in which prior trends do not hold and differences-in-differences comes out wrong

Group 1 is treated after `Time = 0` with a true effect of 1, but it also grows 0.1 faster per unit of time over the whole window. The prior-trends test looks at `Group:Time` before `Time = 0`.

DiD = EFFECT + SLOPE_GAP × (mean post Time − mean pre Time) ≈ 1 + 0.1 × 10 = 2, so the extra trend gets counted as treatment effect.

In [ ]:
rng = np.random.default_rng(0)

EFFECT = 1          # true treatment effect
SLOPE_GAP = 0.1     # how much faster the treated group grows per unit of time


def make_data(n=10_000):
    time = rng.uniform(-10, 10, n)
    group = (rng.uniform(size=n) < 0.5).astype(int)
    post = (time > 0).astype(int)
    y = (1 + 2 * group + 0.05 * time + SLOPE_GAP * group * time
         + EFFECT * group * post + rng.normal(0, 1, n))
    return pd.DataFrame({"Y": y, "Time": time, "Group": group, "post": post})


df = make_data()

pre = smf.ols("Y ~ Group * Time", df[df["Time"] < 0]).fit()
print(f"Prior-trends test: Group:Time = {pre.params['Group:Time']:.3f},"
      f" t = {pre.tvalues['Group:Time']:.1f}")

did = smf.ols("Y ~ Group * post", df).fit()
time_gap = df.loc[df.post == 1, "Time"].mean() - df.loc[df.post == 0, "Time"].mean()
print(f"True effect {EFFECT}")
print(f"DiD estimate {did.params['Group:post']:.3f} (SE {did.bse['Group:post']:.3f})")
print(f"Formula: {EFFECT} + {SLOPE_GAP} x {time_gap:.2f} = {EFFECT + SLOPE_GAP * time_gap:.3f}")